Hospital AI operations assistant

imagine you work in a large hospital
doctors and nurses chat with an AI assistant
instead of answering everything itself, the assistant can call hospital software whenever needed.

- Tool calling: LLMs are excellent at understanding language and reasoning.
- However, they cannot directly interact with external systems such as databases, APIs, hospital software, airline systems, or banking applications.

tool calling is a mechanism that allows an LLM to request that a predefined python function be executed.

workflow:
- user asks a question
- llm determines whether a tool is required
- llm generates a tool call
- python executes the tool
- tool result is sent back to the llm.
- llm generates the final natural language response.

In [1]:
from openai import OpenAI
import json

In [2]:
client = OpenAI()

In [3]:
# HOSPITAL DATABASE: simulating hospital System
# using python dictionaries

In [4]:
appointments = {
    "P101": {
        "name": "Rahul",
        "doctor": "Dr. Mehta",
        "time": "10:30 AM"
    },
    "P102":{
        "name": "Priya",
        "doctor": "Dr. Sharma",
        "time": "2:00 PM"
    }
}

beds = {
    "ICU": 2,
    "General": 8
}

In [5]:
## python functions/ tools
def check_appointment(patient_id):
    return appointments.get(patient_id, {"error":"Appointment not found"})

def check_bed_availability(ward):
    available = beds.get(ward)
    if available is None:
        return {"error": "Ward not found"}
    return{
        "ward": ward,
        "available_beds":available
    }


In [7]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "check_appointment",
            "description": "Check a patient's appointment details",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "string",
                        "description": "Patient ID"
                    }
                },
                "required": ["patient_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_bed_availability",
            "description": "Check available hospital beds",
            "parameters": {
                "type": "object",
                "properties": {
                    "ward": {
                        "type": "string",
                        "description": "Ward name"
                    }
                },
                "required": ["ward"]
            }
        }
    }
]

In [8]:
messages = [
    {
        "role":"user",
        "content":"When is patient P101's appointment?"
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=TOOLS,
    tool_choice="auto"
)

In [11]:
tool_call = response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name

arguments = json.loads(
    tool_call.function.arguments
)

print("Function:", function_name)
print("Arguments:", arguments)

Function: check_appointment
Arguments: {'patient_id': 'P101'}


In [12]:
if function_name == "check_appointment":
    result = check_appointment(arguments["patient_id"])

elif function_name == "check_bed_availability":
    result = check_bed_availability(arguments["ward"])

print(result)

{'name': 'Rahul', 'doctor': 'Dr. Mehta', 'time': '10:30 AM'}


In [13]:
messages.append(response.choices[0].message)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result)
})

In [15]:
final_response = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages,
    tools=TOOLS
)

print(final_response.choices[0].message.content)

Patient P101 (Rahul) has an appointment with Dr. Mehta at 10:30 AM.
